In [1]:
import torch
import os
import subprocess
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import snapshot_download, login
from datasets import load_dataset
from peft import PeftModel, PeftConfig
import json
import re
from IPython.display import Markdown, display
import numpy as np

In [2]:
MODEL_DIR = "./lora-qwen-atc-finetuned/"
BASE_MODEL_DIR = "/data/shared_LLM_models/Qwen2.5-Math-7B/"
BATCH_SIZE = 5
MAX_NEW_TOKENS = 50
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Extracting Data and formating

In [3]:
# read the data from json file
file_path = 'ownship_prompts_data.jsonl'
with open(file_path, 'r') as f:
    data = [json.loads(line) for line in f]

ds = load_dataset('json', data_files=file_path, split='train').train_test_split(test_size=0.2, seed=42)
test = ds['test']

In [4]:
data[:1]  # Display the first two entries to verify the data structure

[{'system_prompt': 'You are an airspace tactical deconfliction assistant. At each time step, an ownship agent is approaching a bottleneck waypoint, \n    such as merging or intersections points, where other agents (intruders) are approaching as well. Based on the information of the intruders, \n    the ownship should take an action to avoid collisions. The ownship agent only has access to the information of the front intruders, \n    but there might be other intruders behind the ownship. Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by \n    suggesting appropriate speed adjustments. The ownship cannot unnecessarily decelerate since it might occlude the airspace for other agents behind it.\n    Now, given the ownship state: [v, v_min, v_max delta_d1, delta_d2, delta_dwpt1, delta_dwpt2, delta_v1, delta_v2, same_route1, same_route2],\n    v: ownship speed(m/s), \n    v_min: ownship minimum speed(m/s), \n    v_max: ownship maximum speed(m/s),\n    

In [34]:
n = 10 #len(test)
test_ids = np.random.choice(len(data), size=n, replace=False)
test_data = [data[i] for i in test_ids]

system_prompts = [item['system_prompt'] for item in test_data]
user_prompts = [item['user_prompt'] for item in test_data]
expected_responses = [item['prompt_response'] for item in test_data]

expected_actions = []
for response in expected_responses:
    m = re.search(r"\b(Accelerate|Decelerate|Hold)\b", response, flags=re.IGNORECASE)
    action = m.group(1).capitalize() if m else "N/A"
    expected_actions.append(action)
    


In [35]:
expected_actions[:10]  # Display the first five expected actions to verify extraction

['Hold',
 'Decelerate',
 'Decelerate',
 'Decelerate',
 'Hold',
 'Accelerate',
 'Decelerate',
 'Decelerate',
 'Hold',
 'Hold']

In [37]:
# test batch
suffix = " Your output must contain EXACTLY one word and no punctuation. Choose only: Accelerate | Decelerate | Hold"
message_batch = [
    [{
        "role": "system",
        "content": system_prompts[i]
    },
    {
        "role": "user",
        "content": user_prompts[i]+suffix
    }]
    for i in range(len(system_prompts))
]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR, use_fast=False, trust_remote_code=True)

try:
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_DIR,
        trust_remote_code=True,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    model = PeftModel.from_pretrained(
        base,
        MODEL_DIR,
        torch_dtype=torch.float16,
        device_map="auto",
    )
except Exception as e:
    # fallback: load directly from MODEL_DIR
    print("Falling back to loading model directly from MODEL_DIR due to error:", e)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_DIR,
        trust_remote_code=True,
        device_map="auto",
        torch_dtype=torch.float16,
    )
model.eval()
# model.to(DEVICE)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
 

In [9]:
print(model.device, base.device)

cuda:0 cuda:0


In [38]:
rendered_prompts = [
    tokenizer.apply_chat_template(
        messages,
        tokenize=False, add_generation_prompt=True,
    ) for messages in message_batch
]
rendered_prompts[:2]  # Display the first two rendered prompts to verify formatting

['<|im_start|>system\nYou are an airspace tactical deconfliction assistant. At each time step, an ownship agent is approaching a bottleneck waypoint, \n    such as merging or intersections points, where other agents (intruders) are approaching as well. Based on the information of the intruders, \n    the ownship should take an action to avoid collisions. The ownship agent only has access to the information of the front intruders, \n    but there might be other intruders behind the ownship. Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by \n    suggesting appropriate speed adjustments. The ownship cannot unnecessarily decelerate since it might occlude the airspace for other agents behind it.\n    Now, given the ownship state: [v, v_min, v_max delta_d1, delta_d2, delta_dwpt1, delta_dwpt2, delta_v1, delta_v2, same_route1, same_route2],\n    v: ownship speed(m/s), \n    v_min: ownship minimum speed(m/s), \n    v_max: ownship maximum speed(m/s),\n  

In [43]:
enc = tokenizer(
    rendered_prompts,
    return_tensors='pt',
    padding=True,
    truncation=True,
).to(model.device)

gen_kwargs = {
        "max_new_tokens": 50,
        "do_sample": False,
        "temperature": 0.0,
        "eos_token_id": tokenizer.eos_token_id,
        "pad_token_id": tokenizer.pad_token_id,
        "use_cache": True,
    }


with torch.inference_mode():
    out = model.generate(**enc, **gen_kwargs)

input_lens = (enc["input_ids"] != tokenizer.pad_token_id).sum(dim=1)
llm_actions = []
for i in range(len(rendered_prompts)):
    gen_ids = out[i, input_lens[i]:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    display(Markdown(text))
    text_output = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    m = re.search(r"\b(Accelerate|Decelerate|Hold)\b", text_output, flags=re.IGNORECASE)
    action = m.group(1).capitalize() if m else "N/A"
    llm_actions.append(action)

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


The ownship state information is:
    [v, v_min, v_max, d1, d2, delta_dwpt1, delta_dwpt2, delta_v1, delta_v2, same_route1, same_route2] =

ownship state: [24.69, 0.00, 41.16, 2471.96, 1939.14, 2471.9

ownship state: [19.55, 0.00, 41.16, 435.07, nan, 435.07, nan, -9.26

ownship state: [23.15, 0.00, 30.87, 395.58, nan, 173.97, nan, 12.3

The ownship speed is 33.44 m/s, which is within the minimum and maximum speed limits of 0.00 m/s and 41.16 m/s, respectively. 
    The first front intr

The ownship speed is 36.53 m/s, which is within the minimum and maximum speed limits of 0.00 m/s and 41.16 m/s, respectively. Since there are no front intr

ownship state: [24.69, 0.00, 41.16, 2197.89, 1754.91, 2168.2

ownship state: [34.98, 0.00, 41.16, 2641.80, 2145.21, 2641.7

The ownship speed is 34.98 m/s, which is within the minimum and maximum speed limits of 0.00 m/s and 41.16 m/s, respectively. 
    The first front intr

The ownship state information is: [25.21, 0.00, 30.87, nan, nan, nan, nan, nan, nan, nan, nan]. 
    Since there are no

In [44]:
llm_actions

['N/A', 'N/A', 'N/A', 'N/A', 'N/A', 'N/A', 'N/A', 'N/A', 'N/A', 'N/A']